# 01 · Stream TGDataset and keep only what the project needs

This notebook extracts the data for the topic-permeability project **without downloading the 460 GB dataset to disk**.
Each of the four archives on Zenodo (`TGDataset_1..4.tar.gz`, 70.7 GB compressed in total) is read straight from the network:
bytes → gzip → tar → JSON parsed one event at a time → rows written to small Parquet files. Nothing else is stored.

The work happens in `tg_stream.py`, which must sit next to this notebook. The notebook checks the setup, runs a smoke test,
starts the extraction **as a background process** (so the notebook stays free and the run survives a kernel restart),
and monitors progress.

**What gets saved in `OUT_DIR`**

| Folder | One row per | Columns | Used for |
|---|---|---|---|
| `forwards/` | forwarded message, every channel, text and media | `dest_id, msg_id, kind, date, src_id, orig_date` | the forwarding graph |
| `texts/` | forwarded text message posted in an English channel | `dest_id, msg_id, date, src_id, orig_date, text` | BERTopic |
| `channels/` | channel | metadata + message and forward counts, `n_fwd_hidden_src` | descriptives, filters |
| `channel_month/` | channel × month | `n_msgs, n_fwd` | normalising activity (RQ3) |
| `labels/` | — | the four label CSVs from the TGDataset repository | language, topic, conspiracy labels |

`src_id` is empty when Telegram hid the original sender; those forwards are counted in `n_fwd_hidden_src`.

**Leaving it overnight**
- Keep the computer awake (Windows: Power settings → Sleep → Never; macOS: run `caffeinate -i` in a terminal; Linux: `systemd-inhibit`).
- If the run stops (crash, reboot, network down for a long time), just run the *Start* cell again. Finished files are skipped;
  a dropped connection is resumed automatically mid-file.

In [1]:
%pip install -q ijson pyarrow requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1 · Settings and checks

In [1]:
import io, json, re, shutil, subprocess, sys, time
from pathlib import Path

import ijson
import pandas as pd
import pyarrow.dataset as ds
import requests

import tg_stream as tg  # tg_stream.py, next to this notebook

OUT_DIR = Path("tgdata")   # where the extracted files go
PARTS = [1, 2, 3, 4]       # archives to process; they run in parallel, one process each

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("JSON parser backend:", ijson.backend, "(should be yajl2_c; anything else is several times slower)")
print(f"Free disk where the outputs go: {shutil.disk_usage(OUT_DIR).free / 1e9:.0f} GB")

JSON parser backend: yajl2_c (should be yajl2_c; anything else is several times slower)
Free disk where the outputs go: 203 GB


## 2 · Labels

In [2]:
label_dir = tg.download_labels(OUT_DIR)
lang = pd.read_csv(label_dir / "channel_to_language_mapping.csv", sep="\t")
topics = pd.read_csv(label_dir / "ch_to_topic_mapping.csv")
consp = pd.read_csv(label_dir / "conspiracy_channels.csv")

print(f"{len(lang):,} channels with a language label, {(lang.language == 'en').sum():,} English")
print(f"{len(topics):,} English channels with a topic label")
print(f"{len(consp):,} channels in the conspiracy list")

120,979 channels with a language label, 19,768 English
19,768 English channels with a topic label
11,618 channels in the conspiracy list


## 3 · Can we reach Zenodo, and does it allow resuming?

In [3]:
for p in PARTS:
    url = tg.PARTS[p]
    try:
        r = requests.get(url, headers={"Range": "bytes=0-0"}, stream=True, timeout=60)
        resumable = r.status_code == 206
        size = r.headers.get("Content-Range", "").split("/")[-1] or r.headers.get("Content-Length", "?")
        r.close()
        print(f"part {p}: HTTP {r.status_code}, {int(size) / 1e9:.1f} GB, resume mid-file: {resumable}")
    except Exception as exc:
        print(f"part {p}: cannot reach Zenodo ({exc})")

part 1: HTTP 206, 19.7 GB, resume mid-file: True
part 2: HTTP 206, 20.4 GB, resume mid-file: True
part 3: HTTP 206, 21.1 GB, resume mid-file: True
part 4: HTTP 206, 9.5 GB, resume mid-file: True


## 4 · Smoke test on the sample file

Parses the 10-channel sample from the TGDataset repository with the same code the full run uses.
Expected: 10 channels, 4,532 messages, 1,285 forwards.

In [4]:
sample_url = "https://raw.githubusercontent.com/SystemsLab-Sapienza/TGDataset/main/tgdataset_sample/tgdataset_preview.json"
raw = requests.get(sample_url, timeout=60).content
english = tg.load_language_ids(label_dir, "en")

test_dir = OUT_DIR / "_smoke_test"
shutil.rmtree(test_dir, ignore_errors=True)
stats = tg.parse_member(io.BytesIO(raw), 0, "sample.json", test_dir, english)
print(stats)
display(pd.read_parquet(test_dir / "channels" / "part0__sample.parquet")
        [["channel_id", "username", "n_text", "n_media", "n_text_fwd", "n_media_fwd", "n_fwd_hidden_src"]])
display(pd.read_parquet(test_dir / "forwards" / "part0__sample.parquet", dtype_backend="numpy_nullable").head())
shutil.rmtree(test_dir)
assert (stats.channels, stats.messages, stats.forwards) == (10, 4532, 1285), "smoke test failed"
print("Smoke test passed.")

MemberStats(channels=10, messages=4532, forwards=1285, texts=719)


,channel_id,username,n_text,n_media,n_text_fwd,n_media_fwd,n_fwd_hidden_src
0,1008738564,CATARACT_BOOKS,10,394,10,301,18
1,1030488269,climagic,400,7,0,0,0
2,1038093473,fuckboycentral,241,77,156,23,0
3,1056181039,liveupdatesyria,266,199,20,30,0
4,1063783344,LawLib,335,238,52,33,0
5,1069292080,foxoblue,101,78,3,14,0
6,1070072600,baeyo,698,218,424,156,0
7,1076104361,DoodleTimeSarah,6,320,0,0,0
8,1076106018,FSAN_Co,255,145,12,8,0
9,1077941735,buhtig,535,9,42,1,0


,dest_id,msg_id,kind,date,src_id,orig_date
0,1008738564,646,text,1637750294,749891673,1637398793
1,1008738564,641,text,1637445253,26811199,1637421802
2,1008738564,640,text,1637270876,1058253025,1637270254
3,1008738564,471,text,1607516955,<NA>,1600260819
4,1008738564,433,text,1563570516,<NA>,1563556602


Smoke test passed.


## 5 · Start the extraction (background)

Starts `tg_stream.py` in its own process: one worker per archive, all in parallel. You can close this notebook;
the run continues. Run this cell again after any interruption: finished files are skipped.

In [5]:
log_dir = OUT_DIR / "_logs"
log_dir.mkdir(parents=True, exist_ok=True)
script = Path(tg.__file__).resolve()

runner_log = open(log_dir / "runner.log", "a", encoding="utf-8")
proc = subprocess.Popen([sys.executable, str(script), str(OUT_DIR.resolve()), *map(str, PARTS)],
                        stdout=runner_log, stderr=subprocess.STDOUT, cwd=script.parent)
(log_dir / "runner.pid").write_text(str(proc.pid))
print(f"Started (pid {proc.pid}). Logs in {log_dir.resolve()}")

Started (pid 21012). Logs in C:\Users\guilh\GitHub\topic-permeability\tgdata\_logs


## 6 · Progress

Run this cell whenever you want an update. `gb_read` is updated each time a file inside the archive finishes
(about every 1,000 channels). `warnings` counts reconnections; a few are normal.

In [22]:
def progress() -> pd.DataFrame:
    rows = []
    for p in PARTS:
        log = OUT_DIR / "_logs" / f"part{p}.log"
        lines = log.read_text(encoding="utf-8").splitlines() if log.exists() else []
        done = [json.loads(f.read_text()) for f in (OUT_DIR / "_done").glob(f"part{p}__*.json")
                if not f.name.endswith("__ALL.json")] if (OUT_DIR / "_done").exists() else []
        gb = next((float(m.group(1)) for line in reversed(lines)
                   if (m := re.search(r"\| ([\d.]+) / ", line))), 0.0)
        finished = (OUT_DIR / "_done" / f"part{p}__ALL.json").exists()
        rows.append({
            "part": p,
            "status": "finished" if finished else ("running" if lines else "not started"),
            "files_done": len(done),
            "gb_read": gb, "gb_total": tg.PART_SIZES_GB[p],
            "pct": round(100 * gb / tg.PART_SIZES_GB[p]),
            "channels": sum(d["channels"] for d in done),
            "messages": sum(d["messages"] for d in done),
            "forwards": sum(d["forwards"] for d in done),
            "english_texts": sum(d["texts"] for d in done),
            "warnings": sum("WARNING" in l for l in lines),
            "last_update": lines[-1][:19] if lines else "",
        })
    return pd.DataFrame(rows).set_index("part")

try:
    state = "running" if proc.poll() is None else f"stopped (exit code {proc.returncode})"
except NameError:
    state = "started in an earlier session"
print("Runner:", state)
progress()

Runner: stopped (exit code 0)


,status,files_done,gb_read,gb_total,pct,channels,messages,forwards,english_texts,warnings,last_update
part,,,,,,,,,,,
1,finished,35,19.71,19.7,100,35000,141037546,26446505,4049049,1,2026-10-11 04:41:35
2,finished,35,20.44,20.4,100,35000,146123023,26448095,3622830,1,2026-10-11 05:03:39
3,finished,35,21.05,21.1,100,35000,143339512,27520167,4384282,1,2026-10-11 05:13:13
4,finished,16,9.49,9.5,100,15979,67820516,13006253,2086544,1,2026-10-11 02:21:57


In [11]:
# If something looks wrong: the last lines of the runner's output (errors show up here)
print("".join((OUT_DIR / "_logs" / "runner.log").read_text(encoding="utf-8").splitlines(keepends=True)[-20:]))

In [12]:
# To stop the run (it can be resumed later with the Start cell):
# proc.terminate()

## 7 · Check the result (after all parts finish)

Compares the totals with the figures reported for TGDataset (120,979 channels, 498,320,597 messages).

In [24]:
channels = pd.read_parquet(OUT_DIR / "channels")
n_msgs = int((channels.n_text + channels.n_media).sum())
print(f"channels: {len(channels):,} (expected 120,979); duplicates: {channels.channel_id.duplicated().sum()}")
print(f"messages: {n_msgs:,} (expected 498,320,597)")
print(f"labelled channels missing from the archives: {len(set(lang.ch_id) - set(channels.channel_id)):,}")

fwd = ds.dataset(OUT_DIR / "forwards", format="parquet")
n_fwd = fwd.count_rows()
n_hidden = fwd.count_rows(filter=ds.field("src_id").is_null())
n_inside = fwd.count_rows(filter=ds.field("src_id").isin(channels.channel_id.to_list()))
print(f"forwards: {n_fwd:,}; hidden sender: {n_hidden:,} ({n_hidden / n_fwd:.1%}); "
      f"source inside the dataset: {n_inside:,} ({n_inside / n_fwd:.1%})")
print(f"English forwarded texts: {ds.dataset(OUT_DIR / 'texts', format='parquet').count_rows():,}")

sizes = {d.name: sum(f.stat().st_size for f in d.glob('*.parquet')) / 1e9
         for d in OUT_DIR.iterdir() if d.is_dir() and not d.name.startswith('_')}
print("disk used (GB):", {k: round(v, 2) for k, v in sizes.items()})

channels: 120,979 (expected 120,979); duplicates: 0
messages: 498,320,597 (expected 498,320,597)
labelled channels missing from the archives: 0
forwards: 93,421,020; hidden sender: 2,745,146 (2.9%); source inside the dataset: 61,932,243 (66.3%)
English forwarded texts: 14,142,705
disk used (GB): {'channels': 0.02, 'channel_month': 0.01, 'forwards': 1.01, 'labels': 0.0, 'texts': 1.53}


In [25]:
import duckdb
con = duckdb.connect()
F, T, C = "tgdata/forwards/*.parquet", "tgdata/texts/*.parquet", "tgdata/channels/*.parquet"

grafo = con.sql(f"""
    WITH e AS (
        SELECT src_id, dest_id FROM '{F}'
        WHERE src_id IN (SELECT channel_id FROM '{C}') AND src_id <> dest_id
    )
    SELECT
        (SELECT count(*) FROM e) AS encaminhamentos_entre_canais,
        (SELECT count(*) FROM (SELECT DISTINCT src_id, dest_id FROM e)) AS arestas,
        (SELECT count(*) FROM (SELECT src_id AS c FROM e UNION SELECT dest_id FROM e)) AS nos
""").df()

textos = con.sql(f"""
    SELECT
        count(*) AS textos,
        count(DISTINCT md5(text)) AS textos_unicos,
        count(*) FILTER (WHERE src_id IN (SELECT channel_id FROM '{C}') AND src_id <> dest_id) AS textos_com_fonte_no_dataset,
        count(DISTINCT md5(text)) FILTER (WHERE src_id IN (SELECT channel_id FROM '{C}') AND src_id <> dest_id) AS unicos_com_fonte_no_dataset
    FROM '{T}'
""").df()

print(grafo.T.to_string(header=False)); print(textos.T.to_string(header=False))

encaminhamentos_entre_canais  57947677
arestas                        4676642
nos                             120715
textos                       14142705
textos_unicos                 4858981
textos_com_fonte_no_dataset   8657235
unicos_com_fonte_no_dataset   2661571


In [26]:
import duckdb
con = duckdb.connect()
F, T, C = "tgdata/forwards/*.parquet", "tgdata/texts/*.parquet", "tgdata/channels/*.parquet"
L = "tgdata/labels/channel_to_language_mapping.csv"

con.sql(f"""CREATE TEMP TABLE en AS SELECT ch_id AS channel_id
            FROM read_csv('{L}', delim='\t', header=true, all_varchar=true,
                          columns={{'ch_id':'VARCHAR','language':'VARCHAR'}})
            WHERE language = 'en'""")
con.sql(f"""CREATE TEMP TABLE e AS SELECT src_id, dest_id FROM '{F}'
            WHERE src_id IN (SELECT channel_id FROM '{C}') AND src_id <> dest_id""")

def resumo(cond):
    return con.sql(f"""WITH x AS (SELECT * FROM e WHERE {cond})
      SELECT (SELECT count(*) FROM x) AS encaminhamentos,
             (SELECT count(*) FROM (SELECT DISTINCT src_id, dest_id FROM x)) AS arestas,
             (SELECT count(*) FROM (SELECT src_id FROM x UNION SELECT dest_id FROM x)) AS nos""").df()

ENd = "CAST(dest_id AS VARCHAR) IN (SELECT channel_id FROM en)"
ENs = "CAST(src_id AS VARCHAR) IN (SELECT channel_id FROM en)"
print("destino em inglês (fonte qualquer):"); print(resumo(ENd).T.to_string(header=False))
print("fonte e destino em inglês:");         print(resumo(f"{ENd} AND {ENs}").T.to_string(header=False))
print(con.sql(f"""SELECT count(*) AS textos_en_en, count(DISTINCT md5(text)) AS unicos_en_en FROM '{T}'
    WHERE src_id <> dest_id AND CAST(src_id AS VARCHAR) IN (SELECT channel_id FROM en)""").df().T.to_string(header=False))

destino em inglês (fonte qualquer):
encaminhamentos  15029602
arestas            896681
nos                 36155
fonte e destino em inglês:
encaminhamentos  13166079
arestas            785846
nos                 18216
textos_en_en  7731006
unicos_en_en  2329663
